[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umangbagadi03/build-your-own-coding-agent/blob/main/notebooks/18_evals.ipynb)

# 18. Evals

**Part 4: Reliability** · about 30 minutes

So far we've judged our agents by running them once and reading the output. That tells you a lot about one run, and almost nothing about the agent. Models are random, so the same agent can fix a bug on Monday and fail on Tuesday. And every change you make, to a prompt, a tool or a model, can make some tasks better and others worse.

An **eval** answers two questions with numbers instead of impressions:

1. **How good is this agent?** Its pass rate on a fixed set of tasks.
2. **Did my change help?** The pass rate before and after, on the same tasks.

This notebook builds a small but real eval suite, and shows the trap that catches most first attempts: grading the agent with tests the agent wrote itself.

**By the end you will have:**

- four bug-fix tasks, each with **hidden tests** the agent never sees
- graders that check the fix, the tests the agent added, and the honesty of its summary
- a report that compares two system prompts
- the maths for judging results when models are random: `pass@k` and `pass^k`

In [1]:
import pathlib, urllib.request

if not pathlib.Path("llm_setup.py").exists():  # Colab opens one notebook at a time, so fetch the helper
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/umangbagadi03/build-your-own-coding-agent/main/notebooks/llm_setup.py", "llm_setup.py"
    )

from llm_setup import connect

llm = connect()

OFFLINE MODE: a scripted local server, not a real model.
The HTTP calls are real. The answers are canned for these lessons.
Token counts are rough estimates and timings are simulated.
Set GROQ_API_KEY (free tier) or LLM_PROVIDER=ollama to use a real model.


In [2]:
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import requests

url = f"{llm.base_url}/chat/completions"

headers = {"Content-Type": "application/json"}
if llm.api_key:
    headers["Authorization"] = f"Bearer {llm.api_key}"

## 1. The tasks

Each task is a small project with a bug, a bug report written the way a user would write it, and a set of **hidden tests**. The agent sees the project and the report. It never sees the hidden tests. We run them only after the agent has finished.

This is the most important design decision in an eval. If you grade with tests the agent can see, or with tests it wrote, you're measuring whether it can make *those* tests pass, which isn't the same thing as fixing the bug.

In [3]:
TASKS = [
    {
        "id": "empty-file",
        "main": "wordcount.py",
        "report": "Bug report: `python wordcount.py empty.txt` crashes on an empty file.",
        "files": {
            "wordcount.py": '"""Print word statistics for a text file."""\n\nimport sys\n\n\ndef stats(text):\n    words = text.split()\n    total_letters = sum(len(word) for word in words)\n    return {"words": len(words), "average_length": round(total_letters / len(words), 2)}\n\n\ndef main(path):\n    with open(path) as f:\n        result = stats(f.read())\n    print(f"{result[\'words\']} words, average length {result[\'average_length\']}")\n\n\nif __name__ == "__main__":\n    main(sys.argv[1])\n',
            "test_wordcount.py": 'import unittest\n\nfrom wordcount import stats\n\n\nclass StatsTests(unittest.TestCase):\n    def test_counts_words(self):\n        self.assertEqual(stats("one two three")["words"], 3)\n',
            "empty.txt": "",
        },
        "hidden_tests": 'import unittest\n\nfrom wordcount import stats\n\n\nclass Hidden(unittest.TestCase):\n    def test_empty(self):\n        self.assertEqual(stats(""), {"words": 0, "average_length": 0.0})\n\n    def test_only_whitespace(self):\n        self.assertEqual(stats("  \\n  ")["words"], 0)\n\n    def test_normal_text_unchanged(self):\n        self.assertEqual(stats("ab abcd")["average_length"], 3.0)\n',
    },
    {
        "id": "boiling-point",
        "main": "convert.py",
        "report": "Bug report: `python convert.py 100` prints 203.0, but water boils at 212 F.",
        "files": {
            "convert.py": '"""Convert Celsius to Fahrenheit."""\n\nimport sys\n\n\ndef to_fahrenheit(celsius):\n    return celsius * 9 / 5 + 23\n\n\nif __name__ == "__main__":\n    print(to_fahrenheit(float(sys.argv[1])))\n',
            "test_convert.py": 'import unittest\n\nfrom convert import to_fahrenheit\n\n\nclass ConvertTests(unittest.TestCase):\n    def test_returns_a_number(self):\n        self.assertIsInstance(to_fahrenheit(10.0), float)\n',
        },
        "hidden_tests": 'import unittest\n\nfrom convert import to_fahrenheit\n\n\nclass Hidden(unittest.TestCase):\n    def test_boiling(self):\n        self.assertEqual(to_fahrenheit(100), 212)\n\n    def test_freezing(self):\n        self.assertEqual(to_fahrenheit(0), 32)\n\n    def test_minus_forty(self):\n        self.assertEqual(to_fahrenheit(-40), -40)\n',
    },
    {
        "id": "double-space",
        "main": "names.py",
        "report": "Bug report: `python names.py 'Asha  Verma'` (two spaces between the names) crashes.",
        "files": {
            "names.py": '"""Turn a full name into initials."""\n\nimport sys\n\n\ndef initials(full_name):\n    """\'Asha Rani Verma\' becomes \'A.R.V.\'"""\n    return ".".join(part[0].upper() for part in full_name.split(" ")) + "."\n\n\nif __name__ == "__main__":\n    print(initials(sys.argv[1]))\n',
            "test_names.py": 'import unittest\n\nfrom names import initials\n\n\nclass NameTests(unittest.TestCase):\n    def test_three_names(self):\n        self.assertEqual(initials("Asha Rani Verma"), "A.R.V.")\n',
        },
        "hidden_tests": 'import unittest\n\nfrom names import initials\n\n\nclass Hidden(unittest.TestCase):\n    def test_double_space(self):\n        self.assertEqual(initials("Asha  Verma"), "A.V.")\n\n    def test_outer_spaces(self):\n        self.assertEqual(initials(" Asha Verma "), "A.V.")\n\n    def test_lower_case(self):\n        self.assertEqual(initials("asha rani verma"), "A.R.V.")\n',
    },
    {
        "id": "coupon",
        "main": "discount.py",
        "report": "Bug report: `python discount.py 200 10` prints 190.0. A 10% coupon on Rs 200 should leave Rs 180.",
        "files": {
            "discount.py": '"""Coupon maths for the shop."""\n\nimport sys\n\n\ndef apply_coupon(price, percent):\n    """Price after a percent-off coupon."""\n    return round(price - percent, 2)\n\n\nif __name__ == "__main__":\n    print(apply_coupon(float(sys.argv[1]), float(sys.argv[2])))\n',
            "test_discount.py": 'import unittest\n\nfrom discount import apply_coupon\n\n\nclass CouponTests(unittest.TestCase):\n    def test_no_coupon(self):\n        self.assertEqual(apply_coupon(100, 0), 100)\n',
        },
        "hidden_tests": 'import unittest\n\nfrom discount import apply_coupon\n\n\nclass Hidden(unittest.TestCase):\n    def test_report_example(self):\n        self.assertEqual(apply_coupon(200, 10), 180)\n\n    def test_other_price(self):\n        self.assertEqual(apply_coupon(500, 10), 450)\n\n    def test_half_off(self):\n        self.assertEqual(apply_coupon(80, 50), 40)\n',
    },
]

for task in TASKS:
    print(f"{task['id']:<15}{task['report']}")

empty-file     Bug report: `python wordcount.py empty.txt` crashes on an empty file.
boiling-point  Bug report: `python convert.py 100` prints 203.0, but water boils at 212 F.
double-space   Bug report: `python names.py 'Asha  Verma'` (two spaces between the names) crashes.
coupon         Bug report: `python discount.py 200 10` prints 190.0. A 10% coupon on Rs 200 should leave Rs 180.


The bugs are small on purpose: one wrong number, a crash on empty input, an unexpected double space, the wrong formula. Small tasks run quickly and cheaply, so you can run the whole suite after every change. A suite you run often beats an impressive one you run once a month.

## 2. The agent under test

Agent v2's working method, with a compact toolbox. One improvement over earlier notebooks: `run_command` doesn't use a shell at all. It splits the command into words itself and only runs Python, which removes the whole class of shell tricks from notebook 08 instead of guarding against them.

In [4]:
def make_tools(workspace):
    def safe(path):
        full = (workspace / path).resolve()
        if not full.is_relative_to(workspace):
            raise PermissionError(f"{path!r} is outside the workspace")
        return full

    def run_command(command):
        try:
            words = shlex.split(command)
        except ValueError as error:
            return f"Error: couldn't parse the command ({error})"
        is_script = len(words) >= 2 and words[0] in ("python", "python3") and words[1].endswith(".py")
        is_tests = words[:3] in (["python", "-m", "unittest"], ["python3", "-m", "unittest"])
        if not (is_script or is_tests):
            return "Denied: only `python <script>.py ...` and `python -m unittest ...` are allowed here."
        env = {k: os.environ[k] for k in ("PATH", "SYSTEMROOT") if k in os.environ}
        try:
            done = subprocess.run([sys.executable, "-B", *words[1:]], cwd=workspace, env=env,
                                  capture_output=True, text=True, timeout=30)
        except subprocess.TimeoutExpired:
            return "stopped: took longer than 30 seconds"
        return f"exit code: {done.returncode}\n" + (done.stdout + done.stderr).replace(str(workspace), ".")[-2000:]

    def read_file(path):
        return "\n".join(f"{n:>4} | {line}" for n, line in enumerate(safe(path).read_text().splitlines(), 1))

    def edit_file(path, old, new):
        text = safe(path).read_text()
        if text.count(old) != 1:
            return f"Error: old must appear exactly once; it appears {text.count(old)} times."
        safe(path).write_text(text.replace(old, new, 1))
        return f"Edited {path}."

    def create_file(path, content):
        if safe(path).exists():
            return f"Error: {path} already exists. Choose a new file name."
        safe(path).write_text(content)
        return f"Created {path}."

    return {"run_command": run_command, "read_file": read_file, "edit_file": edit_file, "create_file": create_file}


def schema(name, description, **props):
    return {"type": "function", "function": {"name": name, "description": description, "parameters": {
        "type": "object", "properties": {k: {"type": "string", "description": v} for k, v in props.items()},
        "required": list(props)}}}


TOOLS = [
    schema("run_command", "Run `python <script>.py args` or `python -m unittest` in the project. "
                          "Output starts with the exit code.", command="The command"),
    schema("read_file", "Read a file with line numbers.", path="File path"),
    schema("edit_file", "Replace text that appears exactly once in a file. No line numbers in old.",
           path="File path", old="Exact text to replace", new="Replacement"),
    schema("create_file", "Create a new file. Fails if it exists.", path="File path", content="File content"),
]

PROMPT_A = """You fix bugs in small Python projects.
1. Reproduce the bug with the command from the report.
2. Read the code and fix it with edit_file.
3. Run the command again to confirm.
4. Add a test for the bug in a new file, then run all tests with `python -m unittest -q`.
Finish with a short summary."""

In [5]:
def run_agent(task, system_prompt, max_steps=15):
    workspace = Path(tempfile.mkdtemp(prefix=f"eval-{task['id']}-")).resolve()
    for path, text in task["files"].items():
        (workspace / path).write_text(text)
    tools = make_tools(workspace)
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": task["report"] + " Fix it and add a test in a new file."}]
    stats = {"model_calls": 0, "input_tokens": 0, "output_tokens": 0, "seconds": 0.0}
    started = time.perf_counter()
    summary = None
    for _ in range(max_steps):
        payload = {"model": llm.model, "messages": messages, "tools": TOOLS}
        response = requests.post(url, headers=headers, json=payload, timeout=llm.timeout)
        response.raise_for_status()
        data = response.json()
        message = data["choices"][0]["message"]
        messages.append(message)
        stats["model_calls"] += 1
        stats["input_tokens"] += (data.get("usage") or {}).get("prompt_tokens", 0)
        stats["output_tokens"] += (data.get("usage") or {}).get("completion_tokens", 0)
        if not message.get("tool_calls"):
            summary = message.get("content") or ""
            break
        for call in message["tool_calls"]:
            name = call["function"]["name"]
            try:
                result = tools[name](**json.loads(call["function"]["arguments"] or "{}")) if name in tools \
                    else f"Error: unknown tool {name!r}"
            except Exception as error:
                result = f"Error: {type(error).__name__}: {error}"
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
    stats["seconds"] = time.perf_counter() - started
    return workspace, summary, stats

## 3. Graders

Each grader answers one yes-or-no question **in code**. Keeping them separate matters. "Failed" on its own tells you nothing, while "fixed the bug, but the test it added doesn't catch it" tells you exactly what to work on.

| Grader | Question |
|---|---|
| `finished` | Did the agent stop on its own, within the step limit? |
| `hidden_tests` | Do the hidden tests pass? **This is the one that counts.** |
| `own_tests_pass` | Do all tests in the project pass, including the agent's? |
| `kept_old_tests` | Are the original test files unchanged? |
| `added_test` | Is there a new test file? |
| `test_catches_bug` | Does the new test fail on the original buggy code? (The mutation check from notebook 10.) |
| `honest_summary` | Does the summary avoid claiming things that didn't happen? |

The last one comes straight from a real run in notebook 10, where a model wrote "I also added a test file" without having added one.

In [6]:
def unittest_passes(workspace):
    done = subprocess.run([sys.executable, "-B", "-m", "unittest", "-q"], cwd=workspace, capture_output=True, text=True,
                          timeout=60, env={k: os.environ[k] for k in ("PATH", "SYSTEMROOT") if k in os.environ})
    return done.returncode == 0


def grade(task, workspace, summary):
    original_tests = [p for p in task["files"] if p.startswith("test_")]
    new_tests = [p.name for p in workspace.glob("test_*.py") if p.name not in task["files"]]
    results = {
        "finished": summary is not None,
        "kept_old_tests": all((workspace / p).read_text() == task["files"][p] for p in original_tests),
        "added_test": bool(new_tests),
        "own_tests_pass": unittest_passes(workspace),
    }

    # The mutation check: put the bug back and make sure the agent's new tests notice.
    main = workspace / task["main"]
    fixed_source = main.read_text()
    main.write_text(task["files"][task["main"]])
    results["test_catches_bug"] = bool(new_tests) and not unittest_passes(workspace)
    main.write_text(fixed_source)

    # Hidden tests, added only now. The name must match unittest's test*.py pattern or they'd never run.
    (workspace / "test_hidden.py").write_text(task["hidden_tests"])
    results["hidden_tests"] = unittest_passes(workspace)
    (workspace / "test_hidden.py").unlink()

    text = (summary or "").lower()
    claims_test = bool(re.search(r"\b(added|wrote|created)\b[^.]*\btests?\b", text))
    claims_pass = bool(re.search(r"\b(tests?|all)\b[^.]*\bpass", text))
    results["honest_summary"] = not (claims_test and not results["added_test"]) and \
                                not (claims_pass and not results["own_tests_pass"])
    return results

Before trusting a grader, test it. A grader that always says "pass" looks exactly like a great agent. Here's the check: on the **unchanged** buggy projects, the hidden tests must fail for every task.

In [7]:
for task in TASKS:
    folder = Path(tempfile.mkdtemp(prefix="grader-check-"))
    for path, text in task["files"].items():
        (folder / path).write_text(text)
    (folder / "test_hidden.py").write_text(task["hidden_tests"])
    print(f"{task['id']:<15}hidden tests on the buggy code: {'pass (BAD)' if unittest_passes(folder) else 'fail (good)'}")
    assert not unittest_passes(folder)
    shutil.rmtree(folder)
print("Checkpoint passed: the hidden tests can tell a buggy project from a fixed one.")

empty-file     hidden tests on the buggy code: fail (good)


boiling-point  hidden tests on the buggy code: fail (good)


double-space   hidden tests on the buggy code: fail (good)


coupon         hidden tests on the buggy code: fail (good)


Checkpoint passed: the hidden tests can tell a buggy project from a fixed one.


This check is worth the extra lines. While writing this notebook, the hidden tests were first saved as `hidden_test.py`. `unittest` only looks for files named `test*.py`, so it silently ran zero hidden tests and every buggy project "passed". This check is what caught it.

A second real bug from writing this notebook is hiding in the `-B` flag on every `python` command above. Without it, Python saves compiled `.pyc` files and reuses them as long as the source file's size and modification time look unchanged. The fix for `convert.py` changes `+ 23` to `+ 32`, so the file stays exactly the same size. The mutation check then rewrites the file within the same second, and Python happily runs the stale compiled version of the *other* file. A correct fix failed its hidden tests until `-B` was added. Graders run code many times in quick succession, which is exactly when caches like this bite.

## 4. Run the suite

In [8]:
def run_suite(system_prompt, label, trials=1):
    rows = []
    for task in TASKS:
        for trial in range(trials):
            workspace, summary, stats = run_agent(task, system_prompt)
            results = grade(task, workspace, summary)
            rows.append({"task": task["id"], "trial": trial, **results, **stats})
            shutil.rmtree(workspace)
            mark = "PASS" if results["hidden_tests"] else "FAIL"
            print(f"  {label}  {task['id']:<15}{mark}  ({stats['model_calls']} calls, {stats['seconds']:.0f}s)")
    return rows


def report(rows, label):
    checks = ["finished", "hidden_tests", "own_tests_pass", "kept_old_tests", "added_test",
              "test_catches_bug", "honest_summary"]
    print(f"\n{label}")
    print(f"{'task':<15}" + "".join(f"{c[:12]:>13}" for c in checks))
    for row in rows:
        print(f"{row['task']:<15}" + "".join(f"{'yes' if row[c] else 'NO':>13}" for c in checks))
    passed = sum(r["hidden_tests"] for r in rows)
    print(f"\nPass rate (hidden tests): {passed}/{len(rows)} = {100 * passed / len(rows):.0f}%")
    print(f"Average model calls: {sum(r['model_calls'] for r in rows) / len(rows):.1f}, "
          f"total input tokens: {sum(r['input_tokens'] for r in rows):,}")


rows_a = run_suite(PROMPT_A, "A")
report(rows_a, "Prompt A")

  A  empty-file     PASS  (7 calls, 3s)


  A  boiling-point  PASS  (7 calls, 3s)


  A  double-space   PASS  (7 calls, 3s)


  A  coupon         FAIL  (7 calls, 3s)

Prompt A
task                finished hidden_tests own_tests_pa kept_old_tes   added_test test_catches honest_summa
empty-file               yes          yes          yes          yes          yes          yes          yes
boiling-point            yes          yes          yes          yes          yes          yes          yes
double-space             yes          yes          yes          yes          yes          yes          yes
coupon                   yes           NO          yes          yes          yes          yes          yes

Pass rate (hidden tests): 3/4 = 75%
Average model calls: 7.0, total input tokens: 22,374


In [9]:
# Checkpoint: the offline script fixes three bugs properly and "fixes" the coupon bug only for the example.
if llm.offline:
    by_task = {r["task"]: r for r in rows_a}
    assert [t for t, r in by_task.items() if not r["hidden_tests"]] == ["coupon"]
    assert by_task["coupon"]["own_tests_pass"] and by_task["coupon"]["test_catches_bug"]
print("Checkpoint passed.")

Checkpoint passed.


Look at the `coupon` row. **Every check the agent could see says yes.** Its own tests pass, its new test catches the bug, the old tests are untouched, and the summary is honest about what happened. Only the hidden tests say no.

Here's what the offline script did. The report said a 10% coupon on Rs 200 should give Rs 180, so it changed `price - percent` to `price - percent * 2`. That works for exactly one example: 10% of 200 happens to be 2 × 10. For Rs 500, it gives 480 instead of 450. The test it wrote checks only the example from the report, so it passes.

This failure is called **overfitting to the example**, and real models do it too, especially when a bug report has one concrete example in it. Without hidden tests, this agent scores 4 out of 4. With them, it scores 3 out of 4, which is the truth.

## 5. Did a change help?

Now the second reason evals exist. Here's a prompt that tries to fix that specific weakness:

In [10]:
PROMPT_B = PROMPT_A.replace(
    "2. Read the code and fix it with edit_file.",
    "2. Read the code and fix the general case with edit_file. The example in the report is one input "
    "among many, so don't write a fix that only works for it. Before editing, think of two other inputs "
    "the fix must handle.",
).replace(
    "4. Add a test for the bug in a new file,",
    "4. Add a test for the bug in a new file, covering the report's example and at least one other input,",
)

rows_b = run_suite(PROMPT_B, "B")
report(rows_b, "Prompt B")

  B  empty-file     PASS  (7 calls, 4s)


  B  boiling-point  PASS  (7 calls, 3s)


  B  double-space   PASS  (7 calls, 3s)


  B  coupon         FAIL  (7 calls, 3s)

Prompt B
task                finished hidden_tests own_tests_pa kept_old_tes   added_test test_catches honest_summa
empty-file               yes          yes          yes          yes          yes          yes          yes
boiling-point            yes          yes          yes          yes          yes          yes          yes
double-space             yes          yes          yes          yes          yes          yes          yes
coupon                   yes           NO          yes          yes          yes          yes          yes

Pass rate (hidden tests): 3/4 = 75%
Average model calls: 7.0, total input tokens: 24,026


**Offline, both prompts score the same,** because the offline script doesn't read the prompt. That's an important result in its own right: an eval tells you what changed, and here nothing could change. On a real model, this is where you find out whether the new wording helps, hurts, or does nothing. Most prompt changes do less than you'd hope, and some do the opposite of what you intended. The only way to know is to measure.

## 6. Randomness: one run proves very little

A real model can pass a task on one run and fail it on the next. So for real comparisons you run each task several times (`trials` in `run_suite`) and look at two numbers:

- **pass@k**: the chance that *at least one* of k attempts succeeds. That's the right number if a human picks the best result, or if the agent can check its own work and retry.
- **pass^k** ("pass all k"): the chance that *all* k attempts succeed. That's the right number for an agent that runs unattended, where every run has to work.

They come apart fast. Here's the same agent seen both ways:

In [11]:
def pass_at_k(p, k):
    return 1 - (1 - p) ** k


def pass_all_k(p, k):
    return p ** k


print(f"{'success rate per run':<22}{'pass@5':>10}{'pass^5':>10}")
for p in [0.5, 0.75, 0.9, 0.99]:
    print(f"{p:<22.0%}{pass_at_k(p, 5):>10.1%}{pass_all_k(p, 5):>10.1%}")

success rate per run      pass@5    pass^5
50%                        96.9%      3.1%
75%                        99.9%     23.7%
90%                       100.0%     59.0%
99%                       100.0%     95.1%


An agent that succeeds 90% of the time almost always gets there in five tries, yet all five runs succeed only 59% of the time. A demo shows you pass@k. Users experience pass^k.

How many trials do you need? With 4 tasks and 1 trial, the difference between 75% and 100% is one task, and that could easily be luck. As a rough guide, you want dozens of task runs before trusting a difference of 10 percentage points, and more for smaller ones. Start small, but don't celebrate a one-task improvement.

## 7. When code can't grade: model judges

Every grader here is code, and that's the ideal. Some qualities can't be checked that way, like "is this summary clear?" or "did it explain the cause correctly?". For those, teams use a second model call as a judge, with a strict rubric:

```text
You are grading a bug-fix summary. Answer only PASS or FAIL.
PASS only if the summary (1) names the actual cause, (2) describes the change made,
and (3) claims nothing that the tool results don't show.
Summary: ...
Tool results: ...
```

Use model judges only where code can't do the job. Check them against your own judgement on a few dozen examples before trusting them, and never let the model being graded grade itself.

## Recap

- An eval is a fixed set of tasks plus graders, run the same way every time.
- **Grade with hidden tests.** An agent's own tests can't catch an agent's own blind spots.
- Split the grade into separate checks, so a failure tells you *what* failed.
- Test your graders. One that always passes looks like a perfect agent.
- Compare changes on the same tasks, with enough trials to beat luck.
- pass@k is what a demo shows. pass^k is what users get.

## Exercises

1. Add a fifth task of your own, with at least three hidden tests. Make sure the hidden tests fail on the buggy code.
2. Run both prompts on a real model with `trials=3`. Is the difference bigger than the run-to-run noise?
3. Add a grader that fails any run using more than 10 model calls. What does it tell you that the pass rate doesn't?
4. Write the model-judge grader from section 7, then find a summary it grades wrong.

**Next:** [19. Cost and Caching](19_cost_and_caching.ipynb). What these runs cost, and how to pay less for the same work.